After loading a series of environments, it can be run directly to generate a webpage. Follow the prompts on the webpage to enter an image, which supports loading and unloading clothes.

You can also choose to comment out the webpage packaging code. Open the last two cells, modify the path of the input image at the instructions in the code, and the result will be saved and displayed in the terminal

In [1]:
# from diffusers import Flux2KleinInpaintPipeline
# print("Flux2KleinInpaintPipeline OK")

In [2]:
# ============================================================
# 挂载 Google Drive
# ============================================================
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ============================================================
# 登录 HuggingFace
# ============================================================
from huggingface_hub import login

HF_TOKEN = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
login(token=HF_TOKEN)

In [4]:
# @title
# ============================================================
# 安装基础依赖和 diffusers 开发版
# ============================================================
!pip uninstall -y diffusers
!pip install -q --no-cache-dir \
  "torch" \
  "triton" \
  # "accelerate>=1.11.0" \
  # "tokenizers>=0.22.0" \
  "sentencepiece" \
  "protobuf<6" \
  "safetensors" \
  "huggingface_hub" \

!pip install -q --no-cache-dir git+https://github.com/huggingface/diffusers.git


Found existing installation: diffusers 0.37.1
Uninstalling diffusers-0.37.1:
  Successfully uninstalled diffusers-0.37.1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.1/509.1 kB 2.2 MB/s eta 0:00:00


In [5]:
# ============================================================
# 安装 Gradio（包装网页）
# ============================================================
!pip install -q gradio

## 安装 CatVTON AutoMasker

In [6]:
# @title
# ===== Install CatVTON AutoMasker for FLUX2_KLEIN try-on mask =====
%cd /content

import os, sys, subprocess, textwrap
from pathlib import Path

CATVTON_DIR = "/content/CatVTON"

# 如果本地没有 CatVTON，就从 GitHub clone；
if not os.path.exists(CATVTON_DIR):
    !git clone -b edited https://github.com/Zheng-Chong/CatVTON.git /content/CatVTON
else:
    print("[INFO] CatVTON already exists:", CATVTON_DIR)

%cd /content/CatVTON

# 2. 安装 CatVTON 需要的基础依赖；
!pip install -q opencv-python einops accelerate huggingface_hub scipy scikit-image tqdm fvcore iopath omegaconf pycocotools

# 安装 DensePose / Detectron2，因为 AutoMasker 需要人体姿态和人体区域信息；
!pip install -q "git+https://github.com/facebookresearch/detectron2.git@main#subdirectory=projects/DensePose"

# 从 HuggingFace 下载 CatVTON 相关 checkpoint；
from huggingface_hub import snapshot_download

CATVTON_CKPT_DIR = snapshot_download(repo_id="zhengchong/CatVTON",local_dir="/content/CatVTON_ckpt",)

print("[INFO] CATVTON_DIR =", CATVTON_DIR)
print("[INFO] CATVTON_CKPT_DIR =", CATVTON_CKPT_DIR)
print("[INFO] DensePose dir =", os.path.join(CATVTON_CKPT_DIR, "DensePose"))
print("[INFO] SCHP dir =", os.path.join(CATVTON_CKPT_DIR, "SCHP"))

# 检查 DensePose 和 SCHP checkpoint 是否存在；
sys.path.insert(0, CATVTON_DIR)

try:
    from model.cloth_masker import AutoMasker
    print("[OK] Imported AutoMasker from model.cloth_masker")

    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"

    automasker = AutoMasker(
        densepose_ckpt=os.path.join(CATVTON_CKPT_DIR, "DensePose"),
        schp_ckpt=os.path.join(CATVTON_CKPT_DIR, "SCHP"),
        device=device,)

    print("[OK] AutoMasker initialized on", device)

except Exception as e:
    print("[ERROR] AutoMasker test failed:")
    print(repr(e))
    raise

/content
Cloning into '/content/CatVTON'...
remote: Enumerating objects: 1358, done.
remote: Counting objects: 100% (309/309), done.
remote: Compressing objects: 100% (181/181), done.
remote: Total 1358 (delta 169), reused 128 (delta 128), pack-reused 1049 (from 2)
Receiving objects: 100% (1358/1358), 16.73 MiB | 29.38 MiB/s, done.
Resolving deltas: 100% (462/462), done.
/content/CatVTON
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

[INFO] CATVTON_DIR = /content/CatVTON
[INFO] CATVTON_CKPT_DIR = /content/CatVTON_ckpt
[INFO] DensePose dir = /content/CatVTON_ckpt/DensePose
[INFO] SCHP dir = /content/CatVTON_ckpt/SCHP
[OK] Imported AutoMasker from model.cloth_masker
[OK] AutoMasker initialized on cuda


In [7]:
# @title
# ============================================================
# 环境检查：确认关键库和模型类可以正常导入
# ============================================================
import torch
import transformers
import diffusers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("diffusers:", diffusers.__version__)

from transformers import Qwen3ForCausalLM, Qwen2TokenizerFast
print("Qwen3ForCausalLM OK")
print("Qwen2TokenizerFast OK")

from transformers import AutoProcessor, AutoModelForCausalLM
print("Florence-2 imports OK: AutoProcessor, AutoModelForCausalLM")

from diffusers import Flux2KleinPipeline
print("Flux2KleinPipeline OK")

from diffusers import Flux2KleinInpaintPipeline
print("Flux2KleinInpaintPipeline OK")

torch: 2.10.0+cu128
transformers: 5.0.0
diffusers: 0.39.0.dev0
Qwen3ForCausalLM OK
Qwen2TokenizerFast OK
Florence-2 imports OK: AutoProcessor, AutoModelForCausalLM


Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Flux2KleinPipeline OK
Flux2KleinInpaintPipeline OK


# Modify the input image path here

In [8]:
# @title
# ============================================================
# 全局配置区：主要修改这里
# ============================================================
import os

# 1. 设置人物图路径 PERSON_PATH；
PERSON_PATH = "/content/drive/MyDrive/DL/TaskB/Designated Test_Character and clothes/Character/30.jpg"
# PERSON_PATH = "/content/drive/MyDrive/DL/TaskB/mode_2.1.jpg"
# PERSON_PATH = "/content/drive/MyDrive/DL/TaskB/model_1.png"
# PERSON_PATH = "/content/drive/MyDrive/DL/TaskB/character/9.jpg"

# 2. 设置服装图路径 CLOTH_PATH；
CLOTH_PATH  = "/content/drive/MyDrive/DL/TaskB/Designated Test_Character and clothes/Clothes/2.jpg"
# CLOTH_PATH  = "/content/drive/MyDrive/DL/TaskB/lower_dress.png"
# CLOTH_PATH  = "/content/drive/MyDrive/DL/TaskB/lower_pan1.png"
# CLOTH_PATH  = "/content/drive/MyDrive/DL/TaskB/lower_pan2.png"

# 3. 设置可选手动 mask 路径 MASK_PATH；
MASK_PATH = None
# MASK_PATH = "/content/drive/MyDrive/DL/TaskB/mask.png"

# 4. 设置输出目录 OUTPUT_DIR；
OUTPUT_DIR = "/content/drive/MyDrive/DL/evalution/output/bighead"

# 5. 设置 FLUX2 推理参数 （1024/1536/2048）
MAX_SIDE = 1536        # 图像最长边尺寸

STEPS = 4           # 采样步数
GUIDANCE_SCALE = 1.0     # 提示词强度
SEED = 353184274509258    # 换装阶段随机种子；
REMOVE_SEED = 1000      # 去衣阶段随机种子；

# 6. 模型选择：
# 1) Colab/T4/A100 更稳：black-forest-labs/FLUX.2-klein-4B
# 2) 更接近工作流的 9B：black-forest-labs/FLUX.2-klein-9B 或 black-forest-labs/FLUX.2-klein-9b-fp8
MODEL_ID = "black-forest-labs/FLUX.2-klein-9B"

# 7. 设置 HuggingFace token；
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", None)

# ============================================================
# 8. 服装区域统一配置
# 自动 upper / lower 分类
# ============================================================
AUTO_GARMENT_BOUND = True

# 非 Gradio 脚本模式下，默认使用 MANUAL_GARMENT_BOUND。
MANUAL_GARMENT_BOUND = "upper"

# 直接复用当前 Qwen-VL 模型
GARMENT_BOUND_MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
def classify_garment_bound_qwen(cloth_path, model_id=GARMENT_BOUND_MODEL_ID):
    import gc
    import torch
    from PIL import Image
    from transformers import AutoProcessor, AutoModelForImageTextToText
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32
    img = Image.open(cloth_path).convert("RGB")
    question = (
        "你是虚拟试衣系统里的服装分类器。"
        "请把图片中的主要服装分成 upper 或 lower。"
        "规则如下："
        "上衣、T恤、衬衫、毛衣、卫衣、外套、夹克、连衣裙、one-piece dress 都归为 upper；"
        "裤子、牛仔裤、短裤、半身裙都归为 lower。"
        "如果不确定，但像连体/连衣裙/覆盖上半身为主，也归为 upper。"
        "最终只能输出一个词：upper 或 lower。不要输出别的内容。")
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        torch_dtype=dtype,
        trust_remote_code=True,).to(device)
    messages = [{"role": "user","content": [{"type": "image", "image": img},{"type": "text", "text": question},],}]
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,)
    inputs = processor(
        text=[text],
        images=[img],
        return_tensors="pt",
        padding=True,)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,)
    input_len = inputs["input_ids"].shape[-1]
    answer = processor.batch_decode(
        output_ids[:, input_len:],
        skip_special_tokens=True)[0].strip().lower()
    print("[GARMENT_BOUND RAW ANSWER] =", repr(answer))
    del model, processor
    if device == "cuda":
        torch.cuda.empty_cache()
    gc.collect()
    if "lower" in answer:
        return "lower"
    return "upper"


# if AUTO_GARMENT_BOUND:
#     GARMENT_BOUND = classify_garment_bound_qwen(CLOTH_PATH)
# else:
#     GARMENT_BOUND = MANUAL_GARMENT_BOUND

# print("[AUTO GARMENT_BOUND] =", GARMENT_BOUND)

# Gradio 版本，GARMENT_BOUND 会在每次上传衣服后动态判断
GARMENT_BOUND = MANUAL_GARMENT_BOUND
print("[DEFAULT GARMENT_BOUND] =", GARMENT_BOUND)
###############################################################
# 9. 用 BOUND_PRESETS 统一管理上装和下装的 mask、prompt、CatVTON 参数。
BOUND_PRESETS = {
    "upper": {
        # GroundingDINO / SAM 自动找 mask 时使用
        "auto_mask_prompt": "upper clothing",
        # CatVTON AutoMasker 使用
        "catvton_cloth_type": "upper",
        # CatVTON try-on mask 后处理
        "tryon_mask_expand": 8,
        "tryon_mask_blur": 2,
        # Size-controllable mask
        "use_size_controllable_mask": True,
        "mask_width_scale": 1.00,
        "mask_length_scale": 1.12,
        "mask_max_width_change_px": 36,
        "mask_max_length_up_px": 80,
        "mask_max_length_down_px": 160,
        "mask_torso_inset_ratio": 0.12,
        # Stage A: remove prompt
        "remove_prompt": "仅去除被遮罩区域内原来的上衣/毛衣，保留模特其他服装、身体、姿势、鞋子和背景不变。",
        # Stage B: try-on prompt prefix
        "tryon_prompt_prefix": (
            "让图1模特自然穿上图2的服装。"
            "图2服装的衣长、袖长、下摆位置、版型和廓形必须按图2自身比例决定，"
            "不要按照图1原衣服轮廓或 mask 边界决定。"
            "保留模特1的脸、发型、姿势、身体比例、鞋子和背景不变。"
            "图2服装款式如下："),
        # 如果你另一个版本里有 prompt_focus_cn / prompt_focus_en，也统一放这里。
        "prompt_focus_cn": "领口、肩线、袖长、衣长、下摆位置、版型、廓形",
        "prompt_focus_en": "collar, shoulder line, sleeve length, top length, hemline position, silhouette, and fit",},
    ###
    "lower": {
        "auto_mask_prompt": "lower clothing",
        "catvton_cloth_type": "lower",
        "tryon_mask_expand": 8,
        "tryon_mask_blur": 2,
        "use_size_controllable_mask": True,
        "mask_width_scale": 1.00,
        "mask_length_scale": 1.12,
        "mask_max_width_change_px": 36,
        "mask_max_length_up_px": 80,
        "mask_max_length_down_px": 160,
        "mask_torso_inset_ratio": 0.12,
        "remove_prompt": "仅去除被遮罩区域内原来的下装/裤子/半身裙，保留模特其他服装、身体、姿势、鞋子和背景不变。",
        "tryon_prompt_prefix": (
            "让图1模特自然穿上图2的服装。"
            "图2服装的裤长或裙长、裤脚或裙摆位置、腰线位置、版型和廓形必须按图2自身比例决定，"
            "不要按照图1原衣服轮廓或 mask 边界决定。"
            "保留模特1的脸、发型、姿势、身体比例、鞋子和背景不变。"
            "图2服装款式如下："),
        "prompt_focus_cn": "腰线、裤长、裙长、裤脚位置、裙摆位置、版型、廓形",
        "prompt_focus_en": "waistline, pants length, skirt length, trouser hem position, skirt hem position, silhouette, and fit",}}

BOUND_CFG = BOUND_PRESETS[GARMENT_BOUND]
# ============================================================
# mask 设置：下面不要手动改，统一从 BOUND_CFG 读取
# ============================================================
AUTO_MASK = True
AUTO_MASK_PROMPT = BOUND_CFG["auto_mask_prompt"]
BOX_THRESHOLD = 0.25
TEXT_THRESHOLD = 0.20
MASK_EXPAND = 10
MASK_BLUR = 0
BLEND_BLUR = 8

# ===== CatVTON AutoMasker for Stage-B try-on mask =====
USE_CATVTON_TRYON_MASK = True
CATVTON_DIR = "/content/CatVTON"   # 改成你自己的 CatVTON 路径
CATVTON_CKPT_DIR = "/content/CatVTON_ckpt"

CATVTON_CLOTH_TYPE = BOUND_CFG["catvton_cloth_type"]
TRYON_MASK_EXPAND = BOUND_CFG["tryon_mask_expand"]
TRYON_MASK_BLUR = BOUND_CFG["tryon_mask_blur"]

# ===== Size-controllable mask（参考 SiCo 思路）=====
USE_SIZE_CONTROLLABLE_MASK = BOUND_CFG["use_size_controllable_mask"]

MASK_WIDTH_SCALE = BOUND_CFG["mask_width_scale"]
MASK_LENGTH_SCALE = BOUND_CFG["mask_length_scale"]

MASK_MAX_WIDTH_CHANGE_PX = BOUND_CFG["mask_max_width_change_px"]
MASK_MAX_LENGTH_UP_PX = BOUND_CFG["mask_max_length_up_px"]
MASK_MAX_LENGTH_DOWN_PX = BOUND_CFG["mask_max_length_down_px"]

MASK_TORSO_INSET_RATIO = BOUND_CFG["mask_torso_inset_ratio"]

# 服装去背景：对应工作流 RMBG
USE_RMBG = True


# Qwen3-VL / Florence2 服装描述
AUTO_CAPTION = True
# CAPTION_BACKEND = "florence2"
# CAPTION_MODEL_ID = "microsoft/Florence-2-large"

CAPTION_BACKEND = "qwen"
# CAPTION_MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
CAPTION_MODEL_ID = GARMENT_BOUND_MODEL_ID

CLOTH_DESCRIPTION = ""

# 两阶段:“去除不需要的服装” -> “模特换装”
RUN_REMOVE_STAGE = True
REMOVE_PROMPT = BOUND_CFG["remove_prompt"]
TRYON_PROMPT_PREFIX = BOUND_CFG["tryon_prompt_prefix"]

# PROMPT_FOCUS_CN = BOUND_CFG["prompt_focus_cn"]
# PROMPT_FOCUS_EN = BOUND_CFG["prompt_focus_en"]

# 可选：工作流里有 f2k_consis.safetensors，Colab 只建议在 MODEL_ID 是 9B 时打开
USE_CONSISTENCY_LORA = True
CONSISTENCY_LORA_REPO = "dx8152/Flux2-Klein-9B-Consistency"
CONSISTENCY_LORA_WEIGHT = "Flux2-Klein-9B-consistency-V2.safetensors"
CONSISTENCY_LORA_SCALE = 0.8

USE_CPU_OFFLOAD = True


[DEFAULT GARMENT_BOUND] = upper


In [9]:
# @title
# ============================================================
# 清空输出目录
# ============================================================
import shutil, os

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 核心换装脚本

In [10]:
# @title
%%writefile /content/flux2_klein_tryon_colab.py
import os
import gc
import sys
import argparse
import inspect
from pathlib import Path

import cv2
import numpy as np
import torch
from PIL import Image, ImageFilter
from scipy.ndimage import binary_fill_holes
from huggingface_hub import login

def dbg(*args):
    print("[DEBUG]", *args, flush=True)
# ----------------------------
# Basic image helpers
#    - 负责读取图片、缩放图片、二值化 mask、膨胀 mask、图像融合、生成预览图。
# ----------------------------
# 创建输出目录；如果目录已经存在也不会报错
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

# 读取图片并转成 RGB，保证后续模型输入格式统一
def load_rgb(path):
    return Image.open(path).convert("RGB")

# 读取 mask 并转成灰度图；白色区域表示需要换装/编辑
def load_l_mask(path):
    return Image.open(path).convert("L")

# 把尺寸四舍五入到 16 的倍数，避免 FLUX/VAE 因尺寸不合适报错
def round_to_multiple(x, multiple=16):
    return max(multiple, int(round(x / multiple) * multiple))

# 按最长边缩放图片，并把宽高调整为 16 的倍数
def resize_longest_to_multiple(img, max_side=1024, multiple=16, resample=Image.LANCZOS):
    w, h = img.size
    scale = max_side / max(w, h)
    new_w = round_to_multiple(w * scale, multiple)
    new_h = round_to_multiple(h * scale, multiple)
    return img.resize((new_w, new_h), resample)

# 把服装图等比例缩放到模特图同尺寸画布中
# 这样 FLUX2 多图编辑时，图1和图2尺寸一致，不容易错位
def fit_to_canvas(img, size, bg=(255, 255, 255), resample=Image.LANCZOS):
    """把服装图等比例放进与模特同尺寸的画布，避免 FLUX2 多图编辑时尺寸错位。"""
    img = img.convert("RGB")
    target_w, target_h = size
    w, h = img.size
    scale = min(target_w / w, target_h / h)
    new_w = max(16, int(round(w * scale)))
    new_h = max(16, int(round(h * scale)))
    resized = img.resize((new_w, new_h), resample)
    canvas = Image.new("RGB", size, bg)
    canvas.paste(resized, ((target_w - new_w) // 2, (target_h - new_h) // 2))
    return canvas

# 将灰度 mask 二值化：大于阈值的区域变白，其余变黑
def binarize_mask(mask_pil, threshold=127):
    arr = np.array(mask_pil.convert("L"))
    arr = (arr > threshold).astype(np.uint8) * 255
    return Image.fromarray(arr, mode="L")


# ----------------------------
# 2. Size-controllable mask
#    - 参考 SiCo 的思路，把已有 mask 当作 regular-fit mask；
#    - 通过 width_scale / length_scale 控制衣服区域更宽、更窄、更长或更短；
#    - upper 和 lower 分开处理，避免上衣和下装使用同一套 mask 逻辑。
# ----------------------------
def grow_mask(mask_pil, expand=10, blur=0, fill_holes=True):
    mask = np.array(binarize_mask(mask_pil), dtype=np.uint8)
    if fill_holes:
        mask = binary_fill_holes(mask > 0).astype(np.uint8) * 255
    if expand and expand > 0:
        k = expand * 2 + 1
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        mask = cv2.dilate(mask, kernel, iterations=1)
    out = Image.fromarray(mask, mode="L")
    if blur and blur > 0:
        out = out.filter(ImageFilter.GaussianBlur(blur))
    return out

def morph_mask(mask_arr, radius, mode="dilate"):
    if radius <= 0:
        return mask_arr
    k = radius * 2 + 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    if mode == "dilate":
        return cv2.dilate(mask_arr, kernel, iterations=1)
    elif mode == "erode":
        return cv2.erode(mask_arr, kernel, iterations=1)
    return mask_arr

##
def make_size_controllable_upper_mask(
    mask_pil,
    width_scale=1.0,
    length_scale=1.0,
    max_width_change_px=36,
    max_length_up_px=80,
    max_length_down_px=160,
    torso_inset_ratio=0.12,
):
    """
    把当前 mask 看作 regular-fit mask，
    再按 width_scale / length_scale 调成更大码、更小码、更长或更短。

    width_scale:
        >1.0  更宽松/更大
        <1.0  更修身/更小
    length_scale:
        >1.0  更长
        <1.0  更短
    """
    mask = binarize_mask(mask_pil)
    arr = np.array(mask, dtype=np.uint8)
    H, W = arr.shape

    bbox = mask.getbbox()
    if bbox is None:
        return mask_pil

    x1, y1, x2, y2 = bbox
    bw = x2 - x1
    bh = y2 - y1

    # ---------- Step 1: 先调“宽度/松紧” ----------
    # 用 bbox 宽度的比例变化，转换成形态学半径
    delta_w = int(round((width_scale - 1.0) * bw * 0.5))
    delta_w = int(np.clip(delta_w, -max_width_change_px, max_width_change_px))

    if delta_w > 0:
        arr = morph_mask(arr, delta_w, mode="dilate")
    elif delta_w < 0:
        arr = morph_mask(arr, abs(delta_w), mode="erode")
    # 重新取 bbox
    tmp_mask = Image.fromarray(arr, mode="L")
    bbox = tmp_mask.getbbox()
    if bbox is None:
        return mask_pil
    x1, y1, x2, y2 = bbox
    bw = x2 - x1
    bh = y2 - y1
    # ---------- Step 2: 再调“衣长” ----------
    target_h = int(round(bh * length_scale))
    delta_h = target_h - bh
    delta_h = int(np.clip(delta_h, -max_length_up_px, max_length_down_px))
    # 只改 torso 下摆，不大幅影响两侧手臂
    torso_pad = int(bw * torso_inset_ratio)
    tx1 = int(np.clip(x1 + torso_pad, 0, W - 1))
    tx2 = int(np.clip(x2 - torso_pad, tx1 + 1, W))
    if delta_h > 0:
        # 向下延长下摆
        new_bottom = int(np.clip(y2 + delta_h, y1 + 20, H))
        arr[y2:new_bottom, tx1:tx2] = 255
    elif delta_h < 0:
        # 缩短下摆：只裁 torso 下半部分
        new_bottom = int(np.clip(y2 + delta_h, y1 + 20, H))
        arr[new_bottom:y2, tx1:tx2] = 0

    out = Image.fromarray(arr, mode="L")
    out = binarize_mask(out)
    return out

##
def make_size_controllable_lower_mask(
    mask_pil,
    width_scale=1.0,
    length_scale=1.0,
    max_width_change_px=36,
    max_length_up_px=80,
    max_length_down_px=160,
    torso_inset_ratio=0.12,
):
    """
    lower 专用：
    把当前 mask 看作 regular-fit 的下装 mask，
    按 width_scale / length_scale 调整“下装宽度”和“裤长/裙长”。

    与 upper 的区别：
    - upper 主要改 torso 中间区域的下摆
    - lower 主要改下装整体下边界，不再按上衣 torso 逻辑处理
    """

    mask = binarize_mask(mask_pil)
    arr = np.array(mask, dtype=np.uint8)
    H, W = arr.shape

    bbox = mask.getbbox()
    if bbox is None:
        return mask_pil

    x1, y1, x2, y2 = bbox
    bw = x2 - x1
    bh = y2 - y1

    # ---------- Step 1: 先调“宽度/松紧” ----------
    delta_w = int(round((width_scale - 1.0) * bw * 0.5))
    delta_w = int(np.clip(delta_w, -max_width_change_px, max_width_change_px))

    if delta_w > 0:
        arr = morph_mask(arr, delta_w, mode="dilate")
    elif delta_w < 0:
        arr = morph_mask(arr, abs(delta_w), mode="erode")

    # 重新取 bbox
    tmp_mask = Image.fromarray(arr, mode="L")
    bbox = tmp_mask.getbbox()
    if bbox is None:
        return mask_pil

    x1, y1, x2, y2 = bbox
    bw = x2 - x1
    bh = y2 - y1

    # ---------- Step 2: 再调“裤长/裙长” ----------
    target_h = int(round(bh * length_scale))
    delta_h = target_h - bh
    delta_h = int(np.clip(delta_h, -max_length_up_px, max_length_down_px))

    # lower 不再只处理中间 torso，
    # 而是基本按整块下装宽度来改下边界。
    # 这里只做一个较小的左右 inset，避免左右边缘过于生硬。
    side_pad = int(bw * torso_inset_ratio * 0.5)
    lx1 = int(np.clip(x1 + side_pad, 0, W - 1))
    lx2 = int(np.clip(x2 - side_pad, lx1 + 1, W))

    if delta_h > 0:
        # 向下延长裤脚 / 裙摆
        new_bottom = int(np.clip(y2 + delta_h, y1 + 20, H))
        arr[y2:new_bottom, lx1:lx2] = 255

    elif delta_h < 0:
        # 缩短裤长 / 裙长：裁掉底部
        new_bottom = int(np.clip(y2 + delta_h, y1 + 20, H))
        arr[new_bottom:y2, lx1:lx2] = 0

    out = Image.fromarray(arr, mode="L")
    out = binarize_mask(out)
    return out

# 如果模特图本身带 alpha 透明通道，则尝试把 alpha 通道当作 mask
def extract_alpha_as_mask(image_path):
    img = Image.open(image_path)
    if img.mode in ("RGBA", "LA"):
        alpha = img.getchannel("A")
        arr = np.array(alpha)
        if arr.max() - arr.min() > 10:
            return alpha
    return None


# 生成 mask 叠加预览图，用来检查需要编辑的区域是否正确
def overlay_mask(image, mask, opacity=0.50, color=(0, 255, 0)):
    image = image.convert("RGB")
    mask = mask.convert("L").resize(image.size, Image.NEAREST)
    img = np.array(image).astype(np.float32)
    m = (np.array(mask).astype(np.float32) / 255.0)[..., None]
    c = np.zeros_like(img)
    c[..., 0] = color[0]
    c[..., 1] = color[1]
    c[..., 2] = color[2]
    out = img * (1 - m * opacity) + c * (m * opacity)
    return Image.fromarray(np.clip(out, 0, 255).astype(np.uint8))


# 按 mask 融合图像：
# mask 白色区域使用 edited 结果
# mask 黑色区域保留 original 原图
# 用于强制保护脸、背景、鞋子等非换装区域
def composite_by_mask(original, edited, mask, blend_blur=8):
    """只把 mask 区域从 FLUX 编辑结果贴回原图，强制保留鞋子、背景、脸等非 mask 区域。"""
    original = original.convert("RGB")
    edited = edited.convert("RGB").resize(original.size, Image.LANCZOS)
    mask = mask.convert("L").resize(original.size, Image.NEAREST)
    if blend_blur and blend_blur > 0:
        mask = mask.filter(ImageFilter.GaussianBlur(blend_blur))
    return Image.composite(edited, original, mask)

# 把多张中间结果拼成一张预览图，方便 Colab 里快速检查效果
def image_grid(images, rows=1, cols=None, bg="white"):
    if cols is None:
        cols = len(images)
    assert rows * cols == len(images)
    images = [im.convert("RGB") for im in images]
    w, h = images[0].size
    canvas = Image.new("RGB", (cols * w, rows * h), bg)
    for i, im in enumerate(images):
        canvas.paste(im.resize((w, h), Image.LANCZOS), ((i % cols) * w, (i // cols) * h))
    return canvas


# ----------------------------
# 3. Mask: manual first, optional GroundingDINO + SAM fallback
#    - 如果没有手动 mask，也没有 alpha mask，就用文字 prompt 自动找衣服区域；
#    - GroundingDINO 负责找框；
#    - SAM 负责根据框生成更精细的 mask；
#    - 最后对 mask 填洞、膨胀、模糊。
# ----------------------------
@torch.no_grad()
def make_grounded_sam_mask(person_img,output_dir,prompt="upper clothing",
    box_threshold=0.25,text_threshold=0.20,expand=10,blur=0,device=None,):
    """没有手动画 mask 时，才用这个自动 mask"""
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, SamModel, SamProcessor

    ensure_dir(output_dir)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    if device == "cpu":
        print("[WARN] 当前是 CPU，GroundingDINO/SAM 会很慢。")

    image = person_img.convert("RGB")

    gd_processor = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-base")
    gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(
        "IDEA-Research/grounding-dino-base").to(device)

    text = prompt.strip()
    if not text.endswith("."):
        text += "."
    inputs = gd_processor(images=image, text=text, return_tensors="pt").to(device)
    gd_outputs = gd_model(**inputs)

    post_fn = gd_processor.post_process_grounded_object_detection
    sig = inspect.signature(post_fn)
    kwargs = {"target_sizes": [image.size[::-1]]}
    if "box_threshold" in sig.parameters:
        kwargs["box_threshold"] = box_threshold
    elif "threshold" in sig.parameters:
        kwargs["threshold"] = box_threshold
    if "text_threshold" in sig.parameters:
        kwargs["text_threshold"] = text_threshold
    try:
        results = post_fn(gd_outputs, inputs.input_ids, **kwargs)[0]
    except TypeError:
        results = post_fn(gd_outputs, **kwargs)[0]

    boxes = results["boxes"]
    scores = results.get("scores", None)
    labels = results.get("labels", None)

    print(f"[GroundingDINO] prompt={prompt!r}, boxes={len(boxes)}")
    if len(boxes) == 0:
        raise RuntimeError(
            "自动 mask 没检测到目标。建议：1）提供手动 mask；2）把 AUTO_MASK_PROMPT 改成 "
            "'shirt'/'sweater'/'dress'/'upper clothing'；3）降低 BOX_THRESHOLD。")

    if scores is not None:
        for i, box in enumerate(boxes[:8]):
            lab = labels[i] if labels is not None else ""
            print(f"  box[{i}] score={float(scores[i]):.3f} label={lab} xyxy={box.detach().cpu().numpy().round(1).tolist()}")

    sam_processor = SamProcessor.from_pretrained("facebook/sam-vit-huge")
    sam_model = SamModel.from_pretrained("facebook/sam-vit-huge").to(device)

    input_boxes = [boxes.detach().cpu().tolist()]
    sam_inputs = sam_processor(image, input_boxes=input_boxes, return_tensors="pt").to(device)
    sam_outputs = sam_model(**sam_inputs)
    post_masks = sam_processor.image_processor.post_process_masks(
        sam_outputs.pred_masks.detach().cpu(),
        sam_inputs["original_sizes"].detach().cpu(),
        sam_inputs["reshaped_input_sizes"].detach().cpu(),)[0]
    iou_scores = sam_outputs.iou_scores.detach().cpu()[0]

    h, w = image.size[1], image.size[0]
    combined = np.zeros((h, w), dtype=bool)
    for i in range(post_masks.shape[0]):
        best_j = int(torch.argmax(iou_scores[i]).item())
        combined |= post_masks[i, best_j].numpy().astype(bool)

    raw = Image.fromarray((combined.astype(np.uint8) * 255), mode="L")
    mask = grow_mask(raw, expand=expand, blur=blur, fill_holes=True)

    raw.save(os.path.join(output_dir, "mask_auto_sam_raw.png"))
    mask.save(os.path.join(output_dir, "mask_auto_grown.png"))

    del gd_model, gd_processor, sam_model, sam_processor
    torch.cuda.empty_cache()
    gc.collect()
    return mask

# 获取最终用于换装的 mask  (优先使用手动 mask；如果没有，则尝试使用图片 alpha 通道；再没有才使用 GroundingDINO + SAM 自动生成)
def get_workflow_mask(person_path, manual_mask_path, person_resized, output_dir, args):
    dbg("[MASK] enter get_workflow_mask")
    dbg("[MASK] manual_mask_path =", manual_mask_path)

    if manual_mask_path and os.path.exists(manual_mask_path):
        dbg("[MASK] using manual MASK_PATH")
        mask = load_l_mask(manual_mask_path)
        mask = resize_longest_to_multiple(mask, args.max_side, 16, Image.NEAREST)

    else:
        alpha = extract_alpha_as_mask(person_path)
        if alpha is not None:
            dbg("[MASK] using alpha channel from PERSON_PATH")
            mask = resize_longest_to_multiple(alpha, args.max_side, 16, Image.NEAREST)

        elif args.auto_mask:
            dbg("[MASK] no manual mask, using GroundingDINO + SAM fallback")
            mask = make_grounded_sam_mask(
                person_img=person_resized,
                output_dir=output_dir,
                prompt=args.auto_mask_prompt,
                box_threshold=args.box_threshold,
                text_threshold=args.text_threshold,
                expand=args.mask_expand,
                blur=args.mask_blur,)
        else:
            raise RuntimeError("没有可用 mask。请设置 MASK_PATH，或把 AUTO_MASK=True。")

    mask = mask.resize(person_resized.size, Image.NEAREST)
    mask = grow_mask(mask, expand=args.mask_expand, blur=args.mask_blur, fill_holes=True)

    # ===== size-controllable mask =====
    if args.use_size_controllable_mask:
        dbg("[MASK] applying size-controllable mask")
        dbg("[MASK] garment_bound =", args.garment_bound)
        dbg("[MASK] width_scale   =", args.mask_width_scale)
        dbg("[MASK] length_scale  =", args.mask_length_scale)

        if args.garment_bound == "lower":
            mask = make_size_controllable_lower_mask(
                mask,
                width_scale=args.mask_width_scale,
                length_scale=args.mask_length_scale,
                max_width_change_px=args.mask_max_width_change_px,
                max_length_up_px=args.mask_max_length_up_px,
                max_length_down_px=args.mask_max_length_down_px,
                torso_inset_ratio=args.mask_torso_inset_ratio,)
        else:
            # 默认仍然走 upper
            mask = make_size_controllable_upper_mask(mask,
                width_scale=args.mask_width_scale,
                length_scale=args.mask_length_scale,
                max_width_change_px=args.mask_max_width_change_px,
                max_length_up_px=args.mask_max_length_up_px,
                max_length_down_px=args.mask_max_length_down_px,
                torso_inset_ratio=args.mask_torso_inset_ratio,)

        # 再做一次轻微清理
        mask = grow_mask(mask, expand=4, blur=args.mask_blur, fill_holes=True)
        mask.save(os.path.join(output_dir, "mask_size_controlled.png"))

    dbg("[MASK] final mask size =", mask.size)
    dbg("[MASK] final mask bbox =", mask.getbbox())
    return mask

# ----------------------------
# 4. CatVTON AutoMasker -> try-on mask
#    - 生成 Stage B 换装专用 try-on mask；
#    - 这个 mask 更接近虚拟试衣中的 garment-agnostic mask；
#    - 可以避免原图衣服轮廓过强地限制新衣服的衣长、袖长和版型。
# ----------------------------
_CATVTON_AUTOMASKER = None

def _load_catvton_automasker(catvton_dir, catvton_ckpt_dir, device="cuda"):
    global _CATVTON_AUTOMASKER
    if _CATVTON_AUTOMASKER is not None:
        return _CATVTON_AUTOMASKER

    if catvton_dir and os.path.isdir(catvton_dir) and catvton_dir not in sys.path:
        sys.path.insert(0, catvton_dir)

    from model.cloth_masker import AutoMasker

    densepose_ckpt = os.path.join(catvton_ckpt_dir, "DensePose")
    schp_ckpt = os.path.join(catvton_ckpt_dir, "SCHP")

    if not os.path.exists(densepose_ckpt):
        raise FileNotFoundError(f"DensePose checkpoint folder not found: {densepose_ckpt}")
    if not os.path.exists(schp_ckpt):
        raise FileNotFoundError(f"SCHP checkpoint folder not found: {schp_ckpt}")

    automasker = AutoMasker(densepose_ckpt=densepose_ckpt,schp_ckpt=schp_ckpt,device=device,)

    _CATVTON_AUTOMASKER = automasker
    return automasker

def _extract_mask_from_automasker_output(out):
    if isinstance(out, Image.Image):
        return out.convert("L")

    if isinstance(out, np.ndarray):
        if out.dtype != np.uint8:
            out = (out > 0).astype(np.uint8) * 255
        return Image.fromarray(out).convert("L")

    if isinstance(out, dict):
        for k in ["mask","agnostic_mask","cloth_agnostic_mask",
            "upper_mask","person_mask",]:
            if k in out and out[k] is not None:
                return _extract_mask_from_automasker_output(out[k])

        raise RuntimeError(
            f"AutoMasker 返回了 dict，但没找到可识别的 mask key。keys={list(out.keys())}")

    if isinstance(out, (list, tuple)) and len(out) > 0:
        return _extract_mask_from_automasker_output(out[0])

    raise RuntimeError(f"无法解析 AutoMasker 输出类型: {type(out)}")


def get_tryon_mask_catvton(person_resized, output_dir, args):
    """
    Stage-B 专用 try-on mask：
    用 CatVTON AutoMasker 生成 agnostic upper-body mask。
    注意：CatVTON AutoMasker 的 __call__ 通常是:
        automasker(image, mask_type)
    或:
        automasker(image, mask_type="upper")
    不是 cloth_type= / image_path=。
    """
    dbg("[TRYON MASK] using CatVTON AutoMasker")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    automasker = _load_catvton_automasker(args.catvton_dir,args.catvton_ckpt_dir,device=device,)

    ensure_dir(output_dir)

    tmp_person = os.path.join(output_dir, "_person_for_catvton_mask.png")
    person_resized.save(tmp_person)

    out = None
    errors = []

    # CatVTON 官方常见调用方式：
    # automasker(person_image, cloth_type)['mask']
    # cloth_type / mask_type 的值是 upper / lower / overall
    candidates = [
        ("PIL + positional mask_type",
         lambda: automasker(person_resized, args.catvton_cloth_type)),
        ("path + positional mask_type",
         lambda: automasker(tmp_person, args.catvton_cloth_type)),
        ("PIL + keyword mask_type",
         lambda: automasker(person_resized, mask_type=args.catvton_cloth_type)),
        ("path + keyword mask_type",
         lambda: automasker(tmp_person, mask_type=args.catvton_cloth_type)),]
    for name, fn in candidates:
        try:
            dbg("[TRYON MASK] trying AutoMasker call:", name)
            out = fn()
            dbg("[TRYON MASK] AutoMasker call success:", name)
            break
        except Exception as e:
            errors.append((name, repr(e)))
            dbg("[TRYON MASK] AutoMasker call failed:", name, repr(e))
    if out is None:
        msg = "\n".join([f"{name}: {err}" for name, err in errors])
        raise RuntimeError("CatVTON AutoMasker 调用失败。所有尝试方式都失败：\n" + msg)

    raw_mask = _extract_mask_from_automasker_output(out)
    raw_mask = raw_mask.resize(person_resized.size, Image.NEAREST)
    raw_mask = binarize_mask(raw_mask)

    tryon_mask = grow_mask(raw_mask,expand=args.tryon_mask_expand,blur=args.tryon_mask_blur,fill_holes=True,)

    raw_mask.save(os.path.join(output_dir, "mask_tryon_catvton_raw.png"))
    tryon_mask.save(os.path.join(output_dir, "mask_tryon_catvton.png"))

    overlay_mask(person_resized, raw_mask).save(os.path.join(output_dir, "mask_tryon_catvton_raw_overlay.png"))
    overlay_mask(person_resized, tryon_mask).save(os.path.join(output_dir, "mask_tryon_catvton_overlay.png"))

    dbg("[TRYON MASK] done")
    dbg("[TRYON MASK] raw bbox =", raw_mask.getbbox())
    dbg("[TRYON MASK] final bbox =", tryon_mask.getbbox())

    return tryon_mask

# ----------------------------
# 5. RMBG and optional garment caption
#    - 对服装图去背景；
#    - 根据 alpha 通道裁掉多余空白；
#    - 把服装放到白底图上，让 FLUX 更关注衣服本身。
# ----------------------------
def remove_garment_background(garment_img, output_dir, enabled=True):
    dbg("[RMBG] enabled =", enabled)
    if not enabled:
        dbg("[RMBG] skipped")
        return garment_img.convert("RGB")
    try:
        from rembg import remove, new_session
        dbg("[RMBG] removing background...")
        session = new_session("isnet-general-use")

        rgba = remove(garment_img.convert("RGBA"), session=session)

        # 用 alpha 通道找衣服主体 bbox，裁掉多余背景空白
        alpha = rgba.getchannel("A")
        bbox = alpha.getbbox()

        if bbox is not None:
            x1, y1, x2, y2 = bbox
            pad = int(0.05 * max(x2 - x1, y2 - y1))
            x1 = max(0, x1 - pad)
            y1 = max(0, y1 - pad)
            x2 = min(rgba.width, x2 + pad)
            y2 = min(rgba.height, y2 + pad)
            rgba = rgba.crop((x1, y1, x2, y2))

        bg = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
        bg.alpha_composite(rgba)
        out = bg.convert("RGB")


        out.save(os.path.join(output_dir, "garment_rmbg_raw.png"))
        dbg("[RMBG] done, saved garment_rmbg_raw.png")
        return out
    except Exception as e:
        dbg("[RMBG] failed, fallback to original garment image:", repr(e))
        return garment_img.convert("RGB")

# 如果用户手动提供服装描述，优先使用手动描述
def _generic_garment_fallback():
    return (
        "the garment shown in image 2, preserving its style, color, material, "
        "print, silhouette, collar, sleeves, hem, and overall design.")


def _save_garment_desc(output_dir, text):
    path = os.path.join(output_dir, "garment_description.txt")
    with open(path, "w", encoding="utf-8") as f:
        f.write(text.strip())
    return path

#    - 否则可以用 Qwen3-VL 或 Florence-2 自动描述服装；
#    - 如果自动描述失败，就用通用 fallback 描述。

# Qwen3-VL
def _describe_garment_qwen(garment_img, model_id, output_dir):
    import torch
    from transformers import AutoProcessor, AutoModelForImageTextToText
    device = "cuda" if torch.cuda.is_available() else "cpu"
    question = (
        "仅描述这件服装本身，不要描述背景、人体、模特或拍摄环境。"
        "重点描述：服装类别、袖长、衣长/裤长、领口、肩线、版型、廓形、下摆、"
        "材质、颜色、图案和装饰细节。"
        "如果是长袖、短袖、无袖、宽松、修身、短款、常规款或长款，必须明确写出来。"
        "如果有特殊版型，衣服款式，如挂脖，连衣裙，也明确写出来"
        "不要猜测品牌，不要加入穿搭建议。150字以内。")
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    # processor = load_qwen_processor(model_id)
    model = AutoModelForImageTextToText.from_pretrained(model_id,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        trust_remote_code=True,).to(device)

    garment_img = garment_img.convert("RGB")

    messages = [{"role": "user","content": [{"type": "image", "image": garment_img},
                {"type": "text", "text": question},],}]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text],images=[garment_img],return_tensors="pt",padding=True,)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = model.generate(**inputs,max_new_tokens=160,do_sample=False,)

    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    desc = generated_text
    _save_garment_desc(output_dir, desc)
    del model, processor
    if device == "cuda":
        torch.cuda.empty_cache()
    return desc

# Florence后端
def _describe_garment_florence2(garment_img, model_id, output_dir):
    import torch
    from transformers import AutoProcessor, AutoModelForCausalLM

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    garment_img = garment_img.convert("RGB")

    # Florence-2 常用 caption task
    task_prompt = "<MORE_DETAILED_CAPTION>"

    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id,torch_dtype=dtype,trust_remote_code=True,).to(device)

    inputs = processor(text=task_prompt,images=garment_img,return_tensors="pt",)

    # 安全搬到设备
    inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = model.generate(input_ids=inputs["input_ids"],pixel_values=inputs["pixel_values"],
                            max_new_tokens=128,do_sample=False,num_beams=3,)

    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]

    # Florence-2 有 post_process_generation
    try:
        parsed = processor.post_process_generation(generated_text,
            task=task_prompt,image_size=(garment_img.width, garment_img.height),)
        if isinstance(parsed, dict) and task_prompt in parsed:
            desc = parsed[task_prompt]
        else:
            desc = generated_text
    except Exception:
        desc = generated_text

    desc = str(desc).strip()

    # 简单清理一下特殊 token
    desc = desc.replace(task_prompt, "").strip()

    _save_garment_desc(output_dir, desc)

    del model, processor
    if device == "cuda":
        torch.cuda.empty_cache()

    return desc


def describe_garment(garment_img,manual_desc,auto_caption,model_id,output_dir,caption_backend="qwen",):
    # 1) 手动描述优先
    if manual_desc is not None and str(manual_desc).strip():
        desc = str(manual_desc).strip()
        _save_garment_desc(output_dir, desc)
        return desc

    # 2) 不启用自动描述 -> 用通用 fallback
    if not auto_caption:
        desc = _generic_garment_fallback()
        _save_garment_desc(output_dir, desc)
        return desc

    # 3) 启用自动描述 -> 按后端分流
    try:
        if caption_backend == "qwen":
            return _describe_garment_qwen(garment_img=garment_img,
                model_id=model_id,output_dir=output_dir,)
        elif caption_backend == "florence2":
            return _describe_garment_florence2(garment_img=garment_img,
                model_id=model_id,output_dir=output_dir,)
        else:
            raise ValueError(f"Unsupported caption_backend: {caption_backend}")

    except Exception as e:
        dbg("[GARMENT CAPTION] failed, fallback to generic description:", repr(e))
        desc = _generic_garment_fallback()
        _save_garment_desc(output_dir, desc)
        return desc

# ----------------------------
# 7. FLUX.2 Klein pipeline
#    - 加载 FLUX2 Klein 模型；
#    - 可选加载 consistency LoRA；
#    - 开启 VAE tiling / slicing / attention slicing / CPU offload 来节省显存。
# ----------------------------

def load_flux2_pipe(model_id, hf_token=None, use_cpu_offload=True, use_lora=False, lora_repo=None, lora_weight=None, lora_scale=0.8):
    dbg("[Flux2] loading model:", model_id)
    dbg("[Flux2] use_cpu_offload =", use_cpu_offload)
    dbg("[Flux2] use_lora =", use_lora)
    dbg("[Flux2] lora_repo =", lora_repo)
    dbg("[Flux2] lora_weight =", lora_weight)
    if hf_token:
        login(token=hf_token, add_to_git_credential=False)

    from diffusers import Flux2KleinPipeline

    print("[Flux2] loading:", model_id)
    pipe = Flux2KleinPipeline.from_pretrained(model_id,torch_dtype=torch.bfloat16,token=hf_token if hf_token else None,)

    if hasattr(pipe, "vae") and pipe.vae is not None:
        try:
            pipe.vae.enable_tiling()
            pipe.vae.enable_slicing()
        except Exception:
            pass
    try:
        pipe.enable_attention_slicing()
    except Exception:
        pass

    if use_lora and lora_repo:
        try:
            print(f"[Flux2] loading LoRA: {lora_repo} / {lora_weight}, scale={lora_scale}")
            if lora_weight:
                pipe.load_lora_weights(lora_repo, weight_name=lora_weight, adapter_name="consistency")
            else:
                pipe.load_lora_weights(lora_repo, adapter_name="consistency")
            pipe.set_adapters(["consistency"], adapter_weights=[lora_scale])
        except Exception as e:
            print("[WARN] LoRA load failed, continue without LoRA. Error:", repr(e))

    if use_cpu_offload:
        print("[Flux2] enable_model_cpu_offload")
        pipe.enable_model_cpu_offload()
    else:
        pipe.to("cuda")

    dbg("[Flux2] pipeline ready")
    return pipe

# ----------------------------
# 8. flux2_edit()
#    - 是真正调用 FLUX2 生成图像的核心函数；
#    - Stage A 用 images=[person]；
#    - Stage B 用 images=[person_for_tryon, garment]；
#    - 会自动适配不同 diffusers 版本里的 image / images / input_images 参数名；
#    - 如果 pipeline 支持 mask_image 或 mask，就传入 mask；
#    - 如果不支持，则后面通过 composite_by_mask 做局部融合。
# ----------------------------
@torch.no_grad()
def flux2_edit(pipe, # 已加载好的 FLUX2 pipeline
        prompt, images, height, width, steps=4, guidance=1.0, seed=1000, mask=None):
    dbg("[Flux2 Edit] enter")
    dbg("prompt =", prompt)
    dbg("height,width =", (height, width))
    dbg("steps,guidance,seed =", (steps, guidance, seed))
    dbg("num images =", len(images) if images is not None else 0)
    for i, im in enumerate(images):
        try:
            dbg(f"image[{i}] size =", im.size)
        except Exception:
            dbg(f"image[{i}] size = <unknown>")
    dbg("mask is None =", mask is None)
    if mask is not None:
        try:
            dbg("mask size =", mask.size)
            dbg("mask bbox =", mask.getbbox())
        except Exception:
            pass

    device = "cuda" if torch.cuda.is_available() else "cpu"
    generator = torch.Generator(device=device).manual_seed(int(seed))

    sig = inspect.signature(pipe.__call__)
    kwargs = {"prompt": prompt,"height": height,"width": width,"guidance_scale": guidance,
        "num_inference_steps": steps,"generator": generator,}
    dbg("[Flux2 Edit] pipe.__call__ parameters =", list(sig.parameters.keys()))
    dbg("[Flux2 Edit] supports mask_image =", "mask_image" in sig.parameters)
    dbg("[Flux2 Edit] supports mask =", "mask" in sig.parameters)
    if "image" in sig.parameters:
        kwargs["image"] = images
    elif "images" in sig.parameters:
        kwargs["images"] = images
    elif "input_images" in sig.parameters:
        kwargs["input_images"] = images
    else:
        kwargs["image"] = images
# 当前 diffusers 版本不支持mask，后面通过 composite_by_mask 做局部融合
    if mask is not None:
        if "mask_image" in sig.parameters:
            kwargs["mask_image"] = mask
            dbg("[Flux2 Edit] PASSING MASK AS mask_image")
        elif "mask" in sig.parameters:
            kwargs["mask"] = mask
            dbg("[Flux2 Edit] PASSING MASK AS mask")
        else:
            dbg("[Flux2 Edit] WARNING: pipeline does NOT support mask_image/mask, mask will NOT be passed")

    has_var_kwargs = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values())
    if not has_var_kwargs:
        kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}

    print(f"[Flux2] edit: size={width}x{height}, steps={steps}, guidance={guidance}, seed={seed}")
    try:
        dbg("[Flux2 Edit] final kwargs keys =", list(kwargs.keys()))
        out = pipe(**kwargs).images[0]
        dbg("[Flux2 Edit] success, out size =", out.size)
        return out
    except TypeError as e:
        dbg("[Flux2 Edit] TypeError:", repr(e))
        if "mask" in str(e).lower():
            kwargs.pop("mask_image", None)
            kwargs.pop("mask", None)
            dbg("[Flux2 Edit] retry without mask")
            out = pipe(**kwargs).images[0]
            dbg("[Flux2 Edit] success without mask, out size =", out.size)
            return out
        raise

# 构造最终换装提示词：
# 包含服装描述 + 保留人物/背景/鞋子的约束 + 只替换 mask 区域的要求
def build_tryon_prompt(prefix, garment_desc):
    return (f"{prefix}{garment_desc}\n"
        "Keep the person identity, face, hair, pose, body shape, hands, legs, shoes, and background unchanged. "
        "The mask is only an editable canvas, not the final clothing outline. "
        "Do not simply fill the entire masked area with clothing. "
        "Infer the natural garment length, hemline position, sleeve length, and silhouette from image 2. "
        "The garment should keep its own original proportions and should not be stretched to match the mask boundary. "
        "Preserve the garment's exact color, material, pattern, print, silhouette, collar, sleeves, hem, and fit. ")


# ----------------------------
# 9. run(args)
#    - 是完整换装流程入口。
#      1) 检查 GPU；
#      2) 读取并缩放人物图；
#      3) 生成 Stage A remove mask；
#      4) 生成 Stage B try-on mask；
#      5) 读取服装图并 RMBG 去背景；
#      6) 生成或读取服装描述；
#      7) 加载 FLUX2 pipeline；
#      8) Stage A 去除原衣服；
#      9) Stage B 参考服装换装；
#      10) 根据 upper/lower 策略做最终融合；
#      11) 保存 result.png、preview、mask、prompt 和调试图。
# ----------------------------
def run(args):
    ensure_dir(args.output_dir)
    dbg("========== RUN START ==========")
    dbg("person_path =", args.person_path)
    dbg("cloth_path  =", args.cloth_path)
    dbg("garment_bound =", args.garment_bound)
    dbg("mask_path   =", args.mask_path)
    dbg("output_dir  =", args.output_dir)
    dbg("model_id    =", args.model_id)
    dbg("max_side    =", args.max_side)
    dbg("steps       =", args.steps)
    dbg("guidance    =", args.guidance_scale)
    dbg("seed        =", args.seed)
    dbg("remove_seed =", args.remove_seed)
    dbg("auto_mask   =", args.auto_mask)
    dbg("use_rmbg    =", args.use_rmbg)
    dbg("run_remove_stage =", args.run_remove_stage)
    dbg("use_cpu_offload  =", args.use_cpu_offload)
    dbg("torch.cuda.is_available() =", torch.cuda.is_available())
    if torch.cuda.is_available():
        dbg("cuda device =", torch.cuda.get_device_name(0))
    if not torch.cuda.is_available():
        raise RuntimeError("FLUX.2 Klein 本地推理需要 CUDA。请在 Colab Runtime 选择 GPU。")
# 读取模特图，并按最长边缩放到 max_side，同时保证尺寸是 16 的倍数
    person_raw = load_rgb(args.person_path)
    person = resize_longest_to_multiple(person_raw, args.max_side, 16, Image.LANCZOS)
    w, h = person.size
    print(f"[Image] person resized: {w}x{h}")
    dbg("[STEP 1] load person image")
    dbg("person_raw size =", person_raw.size)
    dbg("person resized size =", person.size)
# 获取最终换装区域 mask，并保存 mask 原图和叠加预览图
    # Stage-A remove mask：继续用原来的 workflow mask
    dbg("[STEP 2A] get remove mask")
    remove_mask = get_workflow_mask(args.person_path, args.mask_path, person, args.output_dir, args)
    dbg("remove_mask size =", remove_mask.size)
    dbg("remove_mask bbox =", remove_mask.getbbox())
    remove_mask.save(os.path.join(args.output_dir, "mask_used_remove.png"))
    overlay_mask(person, remove_mask).save(os.path.join(args.output_dir, "mask_remove_overlay.png"))
    # Stage-B try-on mask：优先使用 CatVTON AutoMasker
    dbg("[STEP 2B] get try-on mask")
    if args.use_catvton_tryon_mask:
        tryon_mask = get_tryon_mask_catvton(person, args.output_dir, args)
    else:
        tryon_mask = remove_mask
    dbg("tryon_mask size =", tryon_mask.size)
    dbg("tryon_mask bbox =", tryon_mask.getbbox())
    tryon_mask.save(os.path.join(args.output_dir, "mask_used_tryon.png"))
    overlay_mask(person, tryon_mask).save(os.path.join(args.output_dir, "mask_tryon_overlay.png"))

    # 保留一个默认 mask 变量，兼容后面 preview 等旧逻辑
    mask = tryon_mask
# 读取服装图，可选去背景，然后放到与模特图同尺寸的白色画布中
    dbg("[STEP 3] load garment image")
    garment_raw = load_rgb(args.cloth_path)
    dbg("garment_raw size =", garment_raw.size)
    garment_cut = remove_garment_background(garment_raw, args.output_dir, enabled=args.use_rmbg)
    dbg("garment_cut size =", garment_cut.size)
    garment = fit_to_canvas(garment_cut, person.size, bg=(255, 255, 255))
    dbg("garment fitted size =", garment.size)
    garment.save(os.path.join(args.output_dir, "garment_rmbg_fit.png"))
    person.save(os.path.join(args.output_dir, "person_resized.png"))
    dbg("saved garment_rmbg_fit.png")
# 获取服装描述：优先手动描述，其次自动 QwenVL 反推，最后使用通用描述
    dbg("[STEP 4] describe garment")
    garment_desc = describe_garment(
        garment_img=garment,
        manual_desc=args.cloth_description,
        auto_caption=args.auto_caption,
        model_id=args.caption_model_id,
        output_dir=args.output_dir,
        caption_backend=args.caption_backend,
    )
    dbg("garment_desc =", garment_desc)
# 加载 FLUX.2 Klein 图像编辑模型
    dbg("[STEP 5] load FLUX2 pipeline")
    pipe = load_flux2_pipe(
        model_id=args.model_id,
        hf_token=args.hf_token,
        use_cpu_offload=args.use_cpu_offload,
        use_lora=args.use_consistency_lora,
        lora_repo=args.consistency_lora_repo,
        lora_weight=args.consistency_lora_weight,
        lora_scale=args.consistency_lora_scale,
    )
    dbg("[STEP 5 DONE] pipeline loaded")
# Stage A：可选的原衣服去除步骤
# 先让 FLUX2 编辑 mask 区域，把原来的上衣/毛衣去掉
# 再只把 mask 区域融合回原图，作为下一阶段换装输入
    dbg("[STEP 6A] run remove stage")
    dbg("remove_prompt =", args.remove_prompt)
    if args.run_remove_stage:
        removed = flux2_edit(
            pipe=pipe,
            prompt=args.remove_prompt,
            images=[person],
            height=h,
            width=w,
            steps=args.steps,
            guidance=args.guidance_scale,
            seed=args.remove_seed,
            mask=remove_mask,
        )
        dbg("stage_a_remove_raw generated")
        dbg("saving stage_a_remove_raw.png")
        removed.save(os.path.join(args.output_dir, "stage_a_remove_raw.png"))

        person_for_tryon = composite_by_mask(
            person, removed, remove_mask, blend_blur=args.blend_blur
        )
    else:
        person_for_tryon = person
# 构造最终换装 prompt，并保存，方便之后检查模型到底收到什么指令
    tryon_prompt = build_tryon_prompt(args.tryon_prompt_prefix, garment_desc)
    dbg("[STEP 6B] run try-on stage")
    dbg("tryon_prompt =", tryon_prompt)
    Path(args.output_dir, "tryon_prompt.txt").write_text(tryon_prompt, encoding="utf-8")
# Stage B：多图换装
# 输入图1是模特，图2是服装参考图
# FLUX2 根据 prompt 让模特穿上参考服装
    tryon_raw = flux2_edit(pipe=pipe,prompt=tryon_prompt,images=[person_for_tryon, garment],height=h,
        width=w,steps=args.steps,guidance=args.guidance_scale,seed=args.seed,mask=tryon_mask,)
    dbg("stage_b_tryon_raw generated")
    dbg("saving stage_b_tryon_raw.png")
    tryon_raw.save(os.path.join(args.output_dir, "stage_b_tryon_raw.png"))
    # 最终局部融合：只采用生成图中的 mask 区域
    # 非 mask 区域重新mask，既防止胳膊没有，减少身份、背景、鞋子等变化
    dbg("[STEP 7] final composite")
    dbg("garment_bound =", args.garment_bound)
    dbg("blend_blur =", args.blend_blur)

    if args.garment_bound == "lower":
        dbg("[STEP 7] lower mode -> use composite_by_mask")
        final = composite_by_mask(person_for_tryon,tryon_raw,tryon_mask,
            # blend_blur=args.blend_blur,
            blend_blur=3)
    else:
        dbg("[STEP 7] upper mode -> use tryon_raw directly")
        final = tryon_raw
    dbg("final result saved -> result.png")
    # 先保存模型工作尺寸版本（可选）
    final.save(os.path.join(args.output_dir, "result_model_size.png"))

    # 再缩放回原图尺寸，作为最终交付结果
    final_orig = final.resize(person_raw.size, Image.LANCZOS)
    final_orig.save(os.path.join(args.output_dir, "result.png"))

    dbg("final result saved -> result_model_size.png")
    dbg("final result saved -> result.png (original size)")
    # 保存最终结果和中间调试图，方便检查每一步是否正常
    preview = image_grid([person, overlay_mask(person, mask), garment, final],rows=1,cols=4)
    preview.save(os.path.join(args.output_dir, "preview_4panel.png"))

    if args.run_remove_stage:
        preview2 = image_grid([person, overlay_mask(person, mask), person_for_tryon, tryon_raw, final],rows=1,cols=5)
        preview2.save(os.path.join(args.output_dir, "preview_debug.png"))

    dbg("[STEP 8] list output files")
    for fn in sorted(os.listdir(args.output_dir)):
        fp = os.path.join(args.output_dir, fn)
        try:
            dbg(fn, "size =", os.path.getsize(fp))
        except Exception as e:
            dbg(fn, "size check failed:", repr(e))

    print("[Done] saved outputs to:", args.output_dir)
    for name in ["person_resized.png",    "garment_rmbg_fit.png",   "mask_used.png",
            "mask_overlay.png",     "stage_a_remove_raw.png",  "stage_a_remove_fused.png",
            "stage_b_tryon_raw.png",   "result.png",        "preview_4panel.png",
            "preview_debug.png",     "tryon_prompt.txt",]:
        p = os.path.join(args.output_dir, name)
        if os.path.exists(p):

            print(" ", p)

# ----------------------------
# 10. parse_args()
#    - 脚本既可以被 Gradio import 调用，也可以在命令行里直接运行。
# ----------------------------
def parse_args():
    p = argparse.ArgumentParser()
# 输入模特图、服装图、输出目录，以及可选的手动 mask 路径
    p.add_argument("--person_path", required=True)
    p.add_argument("--cloth_path", required=True)
    p.add_argument("--output_dir", required=True)
    p.add_argument("--mask_path", default=None)
# FLUX2 模型和采样参数
# steps 控制推理步数，guidance_scale 控制提示词强度，seed 控制随机性
    p.add_argument("--model_id", default="black-forest-labs/FLUX.2-klein-4B")
    p.add_argument("--hf_token", default=None)
    p.add_argument("--use_cpu_offload", action="store_true")
    p.add_argument("--steps", type=int, default=4)
    p.add_argument("--guidance_scale", type=float, default=1.0)
    p.add_argument("--seed", type=int, default=353184274509258)
    p.add_argument("--remove_seed", type=int, default=1000)
# 图像缩放和 mask 后处理参数
# mask_expand 越大，编辑区域越宽
# blend_blur 越大，最终融合边缘越柔和
    p.add_argument(
        "--garment_bound",
        type=str,
        default="upper",
        choices=["upper", "lower"],
        help="当前服装区域类型：upper 或 lower",
    )
    p.add_argument("--max_side", type=int, default=1024)
    p.add_argument("--use_rmbg", action="store_true")
    p.add_argument("--mask_expand", type=int, default=10)
    p.add_argument("--mask_blur", type=int, default=0)
    p.add_argument("--blend_blur", type=int, default=8)
###
    p.add_argument("--use_catvton_tryon_mask", action="store_true")
    p.add_argument("--catvton_dir", type=str, default="/content/CatVTON")
    p.add_argument("--catvton_ckpt_dir", type=str, default="/content/CatVTON_ckpt")
    p.add_argument("--catvton_cloth_type", type=str, default="upper")
    p.add_argument("--tryon_mask_expand", type=int, default=8)
    p.add_argument("--tryon_mask_blur", type=int, default=2)
# 如果没有手动 mask，可以开启自动 mask
# auto_mask_prompt 用来告诉 GroundingDINO 要找什么区域
    p.add_argument("--auto_mask", action="store_true")
    p.add_argument("--auto_mask_prompt", default="upper clothing")
    p.add_argument("--box_threshold", type=float, default=0.25)
    p.add_argument("--text_threshold", type=float, default=0.20)
    ##
    p.add_argument("--use_size_controllable_mask", action="store_true")

    p.add_argument("--mask_width_scale", type=float, default=1.0)
    p.add_argument("--mask_length_scale", type=float, default=1.0)

    p.add_argument("--mask_max_width_change_px", type=int, default=36)
    p.add_argument("--mask_max_length_up_px", type=int, default=80)
    p.add_argument("--mask_max_length_down_px", type=int, default=160)

    p.add_argument("--mask_torso_inset_ratio", type=float, default=0.12)


# 服装描述：可以手动填写，也可以开启 auto_caption 用 QwenVL 自动反推
    p.add_argument("--cloth_description", type=str, default="")

    p.add_argument(
        "--auto_caption",
        action="store_true",
        help="Use vision-language model to automatically describe the garment image."
    )

    p.add_argument(
        "--caption_backend",
        type=str,
        default="qwen",
        choices=["qwen", "florence2"],
    )

    p.add_argument(
        "--caption_model_id",
        type=str,
        default="Qwen/Qwen3-VL-4B-Instruct",
    )
# 是否先运行原衣服去除阶段
# remove_prompt 控制 Stage A 怎么去除原衣服
# tryon_prompt_prefix 控制 Stage B 怎么描述换装任务
    p.add_argument("--run_remove_stage", action="store_true")
    p.add_argument("--remove_prompt", default="仅去除被遮罩区域内原来的上衣/毛衣，保留模特其他服装、身体、姿势、鞋子和背景不变。")
    p.add_argument("--tryon_prompt_prefix", default="让图1模特穿上图2的服装，版型、尺寸要合理，保留模特1的鞋子不变。图2服装款式如下：")
# 可选加载 consistency LoRA，用来增强 FLUX2 Klein 的一致性
# 如果加载失败，代码会自动跳过，不影响主流程
    p.add_argument("--use_consistency_lora", action="store_true")
    p.add_argument("--consistency_lora_repo", default="dx8152/Flux2-Klein-9B-Consistency")
    p.add_argument("--consistency_lora_weight", default="Flux2-Klein-9B-consistency-V2.safetensors")
    p.add_argument("--consistency_lora_scale", type=float, default=0.8)

    # p.add_argument("--final_mask_expand", type=int, default=24)
    return p.parse_args()

# 脚本入口：读取命令行参数，并启动完整换装流程
if __name__ == "__main__":
    args = parse_args()
    run(args)


Writing /content/flux2_klein_tryon_colab.py


In [11]:
## 包装一个网页

In [ ]:
@title
# ===== Gradio Web UI for FLUX2 Klein Try-On =====
%cd /content

import os
import sys
import uuid
import gc
import importlib
from argparse import Namespace
from PIL import Image

import gradio as gr
import torch

sys.path.append("/content")

import flux2_klein_tryon_colab
importlib.reload(flux2_klein_tryon_colab)

from flux2_klein_tryon_colab import run


GRADIO_OUTPUT_ROOT = os.path.join(OUTPUT_DIR, "gradio_runs")
os.makedirs(GRADIO_OUTPUT_ROOT, exist_ok=True)


def save_uploaded_image(img, path, mode="RGB"):
    if img is None:
        return None
    img = img.convert(mode)
    img.save(path)
    return path


def resolve_bound(cloth_path, bound_mode):
    """
    bound_mode:
    - auto: 用 Qwen 自动判断 upper / lower
    - upper: 手动指定上装
    - lower: 手动指定下装
    """
    if bound_mode == "auto":
        return classify_garment_bound_qwen(cloth_path)
    return bound_mode


def gradio_tryon(
    person_img,
    cloth_img,
    mask_img,
    bound_mode,
    cloth_description,
    auto_caption,
    max_side,
    steps,
    guidance_scale,
    seed,
    remove_seed,
    run_remove_stage,
    use_rmbg,
    use_catvton_mask,
):
    if person_img is None or cloth_img is None:
        raise gr.Error("请上传人物图片和衣服图片。")

    run_id = uuid.uuid4().hex[:8]
    run_dir = os.path.join(GRADIO_OUTPUT_ROOT, run_id)
    os.makedirs(run_dir, exist_ok=True)

    person_path = os.path.join(run_dir, "person_input.png")
    cloth_path = os.path.join(run_dir, "cloth_input.png")
    mask_path = os.path.join(run_dir, "mask_input.png") if mask_img is not None else None

    save_uploaded_image(person_img, person_path, mode="RGB")
    save_uploaded_image(cloth_img, cloth_path, mode="RGB")

    if mask_img is not None:
        save_uploaded_image(mask_img, mask_path, mode="L")

    garment_bound = resolve_bound(cloth_path, bound_mode)
    bound_cfg = BOUND_PRESETS[garment_bound]

    args = Namespace(
        person_path=person_path,
        cloth_path=cloth_path,
        output_dir=run_dir,
        mask_path=mask_path,

        model_id=MODEL_ID,
        hf_token=HF_TOKEN,
        use_cpu_offload=USE_CPU_OFFLOAD,

        steps=int(steps),
        guidance_scale=float(guidance_scale),
        seed=int(seed),
        remove_seed=int(remove_seed),

        garment_bound=garment_bound,
        max_side=int(max_side),

        use_rmbg=bool(use_rmbg),
        mask_expand=MASK_EXPAND,
        mask_blur=MASK_BLUR,
        blend_blur=BLEND_BLUR,

        use_catvton_tryon_mask=bool(use_catvton_mask),
        catvton_dir=CATVTON_DIR,
        catvton_ckpt_dir=CATVTON_CKPT_DIR,
        catvton_cloth_type=bound_cfg["catvton_cloth_type"],
        tryon_mask_expand=bound_cfg["tryon_mask_expand"],
        tryon_mask_blur=bound_cfg["tryon_mask_blur"],

        auto_mask=True,
        auto_mask_prompt=bound_cfg["auto_mask_prompt"],
        box_threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD,

        use_size_controllable_mask=bound_cfg["use_size_controllable_mask"],
        mask_width_scale=bound_cfg["mask_width_scale"],
        mask_length_scale=bound_cfg["mask_length_scale"],
        mask_max_width_change_px=bound_cfg["mask_max_width_change_px"],
        mask_max_length_up_px=bound_cfg["mask_max_length_up_px"],
        mask_max_length_down_px=bound_cfg["mask_max_length_down_px"],
        mask_torso_inset_ratio=bound_cfg["mask_torso_inset_ratio"],

        cloth_description=cloth_description or "",
        auto_caption=bool(auto_caption),
        caption_backend=CAPTION_BACKEND,
        caption_model_id=CAPTION_MODEL_ID,

        run_remove_stage=bool(run_remove_stage),
        remove_prompt=bound_cfg["remove_prompt"],
        tryon_prompt_prefix=bound_cfg["tryon_prompt_prefix"],

        use_consistency_lora=USE_CONSISTENCY_LORA,
        consistency_lora_repo=CONSISTENCY_LORA_REPO,
        consistency_lora_weight=CONSISTENCY_LORA_WEIGHT,
        consistency_lora_scale=CONSISTENCY_LORA_SCALE,
    )

    try:
        run(args)

        result_path = os.path.join(run_dir, "result.png")
        preview_path = os.path.join(run_dir, "preview_4panel.png")
        debug_path = os.path.join(run_dir, "preview_debug.png")
        mask_overlay_path = os.path.join(run_dir, "mask_tryon_overlay.png")
        prompt_path = os.path.join(run_dir, "tryon_prompt.txt")

        result_img = Image.open(result_path).convert("RGB") if os.path.exists(result_path) else None
        preview_img = Image.open(preview_path).convert("RGB") if os.path.exists(preview_path) else None
        debug_img = Image.open(debug_path).convert("RGB") if os.path.exists(debug_path) else None
        mask_overlay_img = Image.open(mask_overlay_path).convert("RGB") if os.path.exists(mask_overlay_path) else None

        if os.path.exists(prompt_path):
            prompt_text = open(prompt_path, "r", encoding="utf-8").read()
        else:
            prompt_text = ""

        info = (
            f"完成。\n"
            f"输出目录：{run_dir}\n"
            f"自动/手动服装区域：{garment_bound}\n\n"
            f"Prompt:\n{prompt_text}"
        )

        return result_img, preview_img, info
        debug_img,
        mask_overlay_img, info

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


with gr.Blocks(title="FLUX2 Klein Virtual Try-On") as demo:
    gr.Markdown("# FLUX2 Klein Virtual Try-On")
    gr.Markdown("上传人物图和衣服图，自动执行完整换装流程。")

    with gr.Row():
        with gr.Column():
            person_input = gr.Image(label="人物图片", type="pil")
            cloth_input = gr.Image(label="衣服图片", type="pil")
            mask_input = gr.Image(label="可选：手动 Mask，白色=重绘区域", type="pil")

            bound_mode = gr.Dropdown(
                choices=["auto", "upper", "lower"],
                value="auto",
                label="服装区域",
            )

            cloth_desc = gr.Textbox(
                label="可选：手动服装描述",
                placeholder="例如：black long-sleeve sweater, loose fit, round collar...",
                lines=3,
            )

            auto_caption_box = gr.Checkbox(
                label="自动生成服装描述 Qwen / Florence",
                value=AUTO_CAPTION,
            )

            run_btn = gr.Button("开始换装", variant="primary")

        with gr.Column():
            result_output = gr.Image(label="最终结果 result.png")
            preview_output = gr.Image(label="四宫格预览 preview_4panel.png")
            # debug_output = gr.Image(label="调试预览 preview_debug.png")
            # mask_output = gr.Image(label="Try-on Mask Overlay")
            info_output = gr.Textbox(label="运行信息 / Prompt", lines=10)

    with gr.Accordion("高级参数", open=False):
        max_side_slider = gr.Slider(768, 2048, value=MAX_SIDE, step=16, label="MAX_SIDE")
        steps_slider = gr.Slider(1, 12, value=STEPS, step=1, label="STEPS")
        guidance_slider = gr.Slider(0.0, 5.0, value=GUIDANCE_SCALE, step=0.1, label="GUIDANCE_SCALE")
        seed_box = gr.Number(value=SEED, label="SEED", precision=0)
        remove_seed_box = gr.Number(value=REMOVE_SEED, label="REMOVE_SEED", precision=0)
        run_remove_box = gr.Checkbox(label="运行 Stage A 去除原衣服", value=RUN_REMOVE_STAGE)
        use_rmbg_box = gr.Checkbox(label="衣服图 RMBG 去背景", value=USE_RMBG)
        use_catvton_box = gr.Checkbox(label="使用 CatVTON Try-on Mask", value=USE_CATVTON_TRYON_MASK)

    run_btn.click(
        fn=gradio_tryon,
        inputs=[
            person_input,
            cloth_input,
            mask_input,
            bound_mode,
            cloth_desc,
            auto_caption_box,
            max_side_slider,
            steps_slider,
            guidance_slider,
            seed_box,
            remove_seed_box,
            run_remove_box,
            use_rmbg_box,
            use_catvton_box,
        ],
        outputs=[
            result_output,
            preview_output,
            # debug_output,
            # mask_output,
            info_output,
        ],
    )

demo.launch(share=True, debug=True)

/content
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c3bb87dfadb3500ff0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

[WARN] LoRA load failed, continue without LoRA. Error: ImportError('Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported')
[Flux2] enable_model_cpu_offload
[DEBUG] [Flux2] pipeline ready
[DEBUG] [STEP 5 DONE] pipeline loaded
[DEBUG] [STEP 6A] run remove stage
[DEBUG] remove_prompt = 仅去除被遮罩区域内原来的上衣/毛衣，保留模特其他服装、身体、姿势、鞋子和背景不变。
[DEBUG] [Flux2 Edit] enter
[DEBUG] prompt = 仅去除被遮罩区域内原来的上衣/毛衣，保留模特其他服装、身体、姿势、鞋子和背景不变。
[DEBUG] height,width = (1536, 848)
[DEBUG] steps,guidance,seed = (4, 1.0, 1000)
[DEBUG] num images = 1
[DEBUG] image[0] size = (848, 1536)
[DEBUG] mask is None = False
[DEBUG] mask size = (848, 1536)
[DEBUG] mask bbox = (147, 256, 687, 1055)
[DEBUG] [Flux2 Edit] pipe.__call__ parameters = ['image', 'prompt', 'height', 'width', 'num_inference_steps', 'sigmas', 'guidance_scale', 'num_images_per_prompt', 'generator', 'latents', 'prompt_embeds', 'negative_prompt_embeds', 'output_type', 'return_dict', 'attention_kwargs', 'callback_

  0%|          | 0/4 [00:00<?, ?it/s]

[DEBUG] [Flux2 Edit] success, out size = (848, 1536)
[DEBUG] stage_a_remove_raw generated
[DEBUG] saving stage_a_remove_raw.png
[DEBUG] [STEP 6B] run try-on stage
[DEBUG] tryon_prompt = 让图1模特自然穿上图2的服装。图2服装的衣长、袖长、下摆位置、版型和廓形必须按图2自身比例决定，不要按照图1原衣服轮廓或 mask 边界决定。保留模特1的脸、发型、姿势、身体比例、鞋子和背景不变。图2服装款式如下：user
仅描述这件服装本身，不要描述背景、人体、模特或拍摄环境。重点描述：服装类别、袖长、衣长/裤长、领口、肩线、版型、廓形、下摆、材质、颜色、图案和装饰细节。如果是长袖、短袖、无袖、宽松、修身、短款、常规款或长款，必须明确写出来。如果有特殊版型，衣服款式，如挂脖，连衣裙，也明确写出来不要猜测品牌，不要加入穿搭建议。150字以内。
assistant
无袖挂脖连衣裙，衣长及膝，下摆不规则开衩。领口为高领挂脖设计，肩线平直。版型宽松垂坠，廓形飘逸。下摆为双层结构，外层为半透明网纱，内层为黑色底布。材质为轻薄网纱，点缀细密亮片。整体为纯黑色，图案为均匀分布的微小亮片装饰，装饰细节精致，富有光泽感。
Keep the person identity, face, hair, pose, body shape, hands, legs, shoes, and background unchanged. The mask is only an editable canvas, not the final clothing outline. Do not simply fill the entire masked area with clothing. Infer the natural garment length, hemline position, sleeve length, and silhouette from image 2. The garment should keep its own original proportions and should not be stretched 

  0%|          | 0/4 [00:00<?, ?it/s]

[DEBUG] [Flux2 Edit] success, out size = (848, 1536)
[DEBUG] stage_b_tryon_raw generated
[DEBUG] saving stage_b_tryon_raw.png
[DEBUG] [STEP 7] final composite
[DEBUG] garment_bound = upper
[DEBUG] blend_blur = 8
[DEBUG] [STEP 7] upper mode -> use tryon_raw directly
[DEBUG] final result saved -> result.png
[DEBUG] final result saved -> result_model_size.png
[DEBUG] final result saved -> result.png (original size)
[DEBUG] [STEP 8] list output files
[DEBUG] _person_for_catvton_mask.png size = 511867
[DEBUG] cloth_input.png size = 281719
[DEBUG] garment_description.txt size = 863
[DEBUG] garment_rmbg_fit.png size = 360147
[DEBUG] mask_auto_grown.png size = 5598
[DEBUG] mask_auto_sam_raw.png size = 5469
[DEBUG] mask_remove_overlay.png size = 480899
[DEBUG] mask_size_controlled.png size = 5861
[DEBUG] mask_tryon_catvton.png size = 14892
[DEBUG] mask_tryon_catvton_overlay.png size = 496093
[DEBUG] mask_tryon_catvton_raw.png size = 4744
[DEBUG] mask_tryon_catvton_raw_overlay.png size = 487384


In [ ]:
# @title
# # @title
# # 运行前检查 + 真正执行脚本
# %cd /content

# import os, shutil, subprocess, shlex, time
# from pathlib import Path
# from PIL import Image
# from IPython.display import display

# print("========== NOTEBOOK DEBUG ==========")
# print("PERSON_PATH =", PERSON_PATH)
# print("CLOTH_PATH  =", CLOTH_PATH)
# print("MASK_PATH   =", MASK_PATH)
# print("OUTPUT_DIR  =", OUTPUT_DIR)
# print("MODEL_ID    =", MODEL_ID)
# print("MAX_SIDE    =", MAX_SIDE)
# print("STEPS       =", STEPS)
# print("GUIDANCE    =", GUIDANCE_SCALE)
# print("SEED        =", SEED)
# print("REMOVE_SEED =", REMOVE_SEED)
# print("AUTO_MASK   =", AUTO_MASK)
# print("USE_RMBG    =", USE_RMBG)
# print("RUN_REMOVE_STAGE =", RUN_REMOVE_STAGE)
# print("USE_CPU_OFFLOAD  =", USE_CPU_OFFLOAD)
# print("AUTO_CAPTION =", AUTO_CAPTION)
# print("CLOTH_DESCRIPTION =", repr(CLOTH_DESCRIPTION))
# print("USE_CATVTON_TRYON_MASK =", USE_CATVTON_TRYON_MASK)
# print("CATVTON_DIR =", CATVTON_DIR)
# print("CATVTON_CLOTH_TYPE =", CATVTON_CLOTH_TYPE)
# print("TRYON_MASK_EXPAND =", TRYON_MASK_EXPAND)
# print("TRYON_MASK_BLUR =", TRYON_MASK_BLUR)
# print()

# # 路径存在性检查
# for p in [PERSON_PATH, CLOTH_PATH]:
#     print(f"[EXISTS] {p} -> {os.path.exists(p)}")
# if MASK_PATH:
#     print(f"[EXISTS] {MASK_PATH} -> {os.path.exists(MASK_PATH)}")
# print()

# # 预览当前输入，防止你以为改了路径但实际上没生效
# print("========== INPUT PREVIEW ==========")
# if os.path.exists(PERSON_PATH):
#     print("[PERSON] size =", Image.open(PERSON_PATH).size)
#     display(Image.open(PERSON_PATH).convert("RGB").resize((256, 256)))
# else:
#     print("[PERSON] missing")

# if os.path.exists(CLOTH_PATH):
#     print("[CLOTH] size =", Image.open(CLOTH_PATH).size)
#     display(Image.open(CLOTH_PATH).convert("RGB").resize((256, 256)))
# else:
#     print("[CLOTH] missing")

# if MASK_PATH and os.path.exists(MASK_PATH):
#     print("[MASK] size =", Image.open(MASK_PATH).size)
#     display(Image.open(MASK_PATH).convert("L").resize((256, 256)))
# print()

# # 可选：每次都清空输出目录，避免看到旧结果
# if os.path.exists(OUTPUT_DIR):
#     print("[INFO] removing old OUTPUT_DIR:", OUTPUT_DIR)
#     shutil.rmtree(OUTPUT_DIR)
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# cmd = [
    # "python", "/content/flux2_klein_tryon_colab.py",
    # "--person_path", PERSON_PATH,
    # "--cloth_path", CLOTH_PATH,
    # "--garment_bound", GARMENT_BOUND,
    # "--output_dir", OUTPUT_DIR,
    # "--model_id", MODEL_ID,
    # "--max_side", str(MAX_SIDE),
    # "--steps", str(STEPS),
    # "--guidance_scale", str(GUIDANCE_SCALE),
    # "--seed", str(SEED),
    # "--remove_seed", str(REMOVE_SEED),
    # "--auto_mask_prompt", AUTO_MASK_PROMPT,
    # "--box_threshold", str(BOX_THRESHOLD),
    # "--text_threshold", str(TEXT_THRESHOLD),

    # "--mask_expand", str(MASK_EXPAND),
    # "--mask_blur", str(MASK_BLUR),
    # "--blend_blur", str(BLEND_BLUR),
    # "--catvton_dir", CATVTON_DIR,
    # "--catvton_ckpt_dir", CATVTON_CKPT_DIR,
    # "--catvton_cloth_type", CATVTON_CLOTH_TYPE,
    # "--tryon_mask_expand", str(TRYON_MASK_EXPAND),
    # "--tryon_mask_blur", str(TRYON_MASK_BLUR),

    # "--caption_backend", CAPTION_BACKEND,
    # "--caption_model_id", CAPTION_MODEL_ID,
    # "--cloth_description", CLOTH_DESCRIPTION,
    # "--remove_prompt", REMOVE_PROMPT,
#     "--tryon_prompt_prefix", TRYON_PROMPT_PREFIX,
#     "--consistency_lora_repo", CONSISTENCY_LORA_REPO,
#     "--consistency_lora_weight", CONSISTENCY_LORA_WEIGHT,
#     "--consistency_lora_scale", str(CONSISTENCY_LORA_SCALE),
# ]
# cmd += [
#     "--mask_width_scale", str(MASK_WIDTH_SCALE),
#     "--mask_length_scale", str(MASK_LENGTH_SCALE),
#     "--mask_max_width_change_px", str(MASK_MAX_WIDTH_CHANGE_PX),
#     "--mask_max_length_up_px", str(MASK_MAX_LENGTH_UP_PX),
#     "--mask_max_length_down_px", str(MASK_MAX_LENGTH_DOWN_PX),
#     "--mask_torso_inset_ratio", str(MASK_TORSO_INSET_RATIO),
# ]

# if MASK_PATH:
#     cmd += ["--mask_path", MASK_PATH]
# if AUTO_MASK:
#     cmd += ["--auto_mask"]
# if USE_RMBG:
#     cmd += ["--use_rmbg"]
# if AUTO_CAPTION:
#     cmd += ["--auto_caption"]
# if RUN_REMOVE_STAGE:
#     cmd += ["--run_remove_stage"]
# if USE_CONSISTENCY_LORA:
#     cmd += ["--use_consistency_lora"]
# if USE_CPU_OFFLOAD:
#     cmd += ["--use_cpu_offload"]
# if USE_CATVTON_TRYON_MASK:
#     cmd += ["--use_catvton_tryon_mask"]
# if HF_TOKEN:
#     cmd += ["--hf_token", HF_TOKEN]

# if USE_SIZE_CONTROLLABLE_MASK:
#     cmd += ["--use_size_controllable_mask"]

# print("\n========== COMMAND ==========")
# print(" ".join(shlex.quote(x) for x in cmd))

# t0 = time.time()
# result = subprocess.run(
#     cmd,
#     text=True,
#     stdout=subprocess.PIPE,
#     stderr=subprocess.PIPE,
# )
# t1 = time.time()

# print("\n========== RETURN CODE ==========")
# print(result.returncode)
# print(f"Elapsed: {t1 - t0:.1f} sec")

# print("\n========== STDOUT ==========")
# print(result.stdout)

# print("\n========== STDERR ==========")
# print(result.stderr)

# print("\n========== OUTPUT FILES ==========")
# if os.path.exists(OUTPUT_DIR):
#     for name in sorted(os.listdir(OUTPUT_DIR)):
#         p = os.path.join(OUTPUT_DIR, name)
#         print(name, " | size =", os.path.getsize(p), " | mtime =", time.ctime(os.path.getmtime(p)))

# if result.returncode != 0:
#     raise RuntimeError("脚本运行失败，上面已经打印了 stdout/stderr")


In [ ]:
# @title
# # @title
# # 显示中间结果和最终结果
# import os, time
# from PIL import Image
# from IPython.display import display

# print("========== OUTPUT CHECK ==========")
# print("OUTPUT_DIR =", OUTPUT_DIR)

# targets = [
#     "person_resized.png",
#     "garment_rmbg_fit.png",
#     "mask_used.png",
#     "mask_overlay.png",
#     "stage_a_remove_raw.png",
#     # "stage_a_remove_fused.png",
#     "stage_b_tryon_raw.png",
#     "result.png",
#     "preview_4panel.png",
#     "preview_debug.png",
# ]

# for name in targets:
#     p = os.path.join(OUTPUT_DIR, name)
#     if os.path.exists(p):
#         print(f"[FOUND] {name} | size={os.path.getsize(p)} | mtime={time.ctime(os.path.getmtime(p))}")
#     else:
#         print(f"[MISS ] {name}")

# def show_img(title, path, size=(256, 256)):
#     if os.path.exists(path):
#         print("\n===", title, "===")
#         img = Image.open(path).convert("RGB")
#         print("size =", img.size)
#         display(img.resize(size))
#     else:
#         print("[missing]", title, path)

# show_img("person_resized", os.path.join(OUTPUT_DIR, "person_resized.png"))
# show_img("garment_rmbg_fit", os.path.join(OUTPUT_DIR, "garment_rmbg_fit.png"))
# show_img("mask_overlay", os.path.join(OUTPUT_DIR, "mask_overlay.png"))
# # show_img("stage_a_remove_fused", os.path.join(OUTPUT_DIR, "stage_a_remove_fused.png"))
# show_img("stage_b_tryon_raw", os.path.join(OUTPUT_DIR, "stage_b_tryon_raw.png"))
# show_img("result", os.path.join(OUTPUT_DIR, "result.png"))
